# Sauvegarde du Modèle avec BentoML

## Objectif
Ce notebook démontre comment sauvegarder et déployer le **modèle Linear Regression gagnant** du projet P6 Seattle Energy Prediction en utilisant **BentoML**.

---

## 1️. Import des Librairies Requises

In [1]:
# Import des librairies essentielles
import pandas as pd
import numpy as np
import bentoml
from datetime import datetime

# Import Scikit-learn
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 2️. Chargement et Préparation des Données

In [2]:
# Chemins des données
data_path = "../sources/2016_Building_Energy_Benchmarking_03_encoded.csv"
model_path = "../models/"

# Charger les données nettoyées
X = pd.read_csv('../sources/2016_Building_Energy_Benchmarking_03X_building_consumption.csv')
y = pd.read_csv('../sources/2016_Building_Energy_Benchmarking_03y_building_consumption.csv')

# Chargement des données d'entraînement
print("\n" + "=" * 100)
print("📊 Chargement des données...")
print("=" * 100 + "\n")

data = pd.read_csv(data_path)
print(f"✅ Données chargées: {data.shape[0]} lignes, {data.shape[1]} colonnes")
print(f"📝 Colonnes disponibles: {list(data.columns)}")



📊 Chargement des données...

✅ Données chargées: 1277 lignes, 43 colonnes
📝 Colonnes disponibles: ['OSEBuildingID', 'PropertyGFATotal', 'NumberofFloors', 'SiteEnergyUse(kWh)', 'SiteEUI(kWh/m2)', 'Electricity(kWh)', 'NaturalGas(kWh)', 'Age_batiment', 'Categorie_age', 'Consommation_par_etage', 'Surface_par_etage', 'Densite_energetique', 'Ratio_elec_gaz', 'Taille_batiment', 'Hauteur_batiment', 'Surface_x_Age', 'Log_SiteEnergyUse', 'PrimaryPropertyType_Hospital', 'PrimaryPropertyType_Hotel', 'PrimaryPropertyType_K-12 School', 'PrimaryPropertyType_Laboratory', 'PrimaryPropertyType_Large Office', 'PrimaryPropertyType_Low-Rise Multifamily', 'PrimaryPropertyType_Medical Office', 'PrimaryPropertyType_Mixed Use Property', 'PrimaryPropertyType_Other', 'PrimaryPropertyType_Refrigerated Warehouse', 'PrimaryPropertyType_Residence Hall', 'PrimaryPropertyType_Restaurant', 'PrimaryPropertyType_Retail Store', 'PrimaryPropertyType_Self-Storage Facility', 'PrimaryPropertyType_Senior Care Community', 'Pri

In [3]:
# Préparation des features et target
features = [
    'PropertyGFATotal', 'NumberofFloors', 
    'Electricity(kWh)', 'NaturalGas(kWh)',
    'PrimaryPropertyType_Large Office', 'PrimaryPropertyType_Other',
    'PrimaryPropertyType_Retail Store', 'PrimaryPropertyType_Small- and Mid-Sized Office',
    'PrimaryPropertyType_Warehouse'
]

target = 'SiteEnergyUse(kWh)'

print("\n" + "=" * 100)
print("Préparation des données d'entraînement...")
print("=" * 100 + "\n")

print(f"Features utilisées: {len(features)}")
print(f"Target: {target}")

# Vérification des colonnes disponibles
available_features = [f for f in features if f in data.columns]
missing_features = [f for f in features if f not in data.columns]

# Utilisation des features disponibles
X = data[available_features].copy()
y = data[target].copy()

# Nettoyage des données
print("\n" + "=" * 100)
print("Nettoyage des données...")
print("=" * 100 + "\n")
print(f"Valeurs manquantes X: {X.isnull().sum().sum()}")
print(f"Valeurs manquantes y: {y.isnull().sum()}")

# Suppression des valeurs manquantes
mask = ~(X.isnull().any(axis=1) | y.isnull())
X = X[mask]
y = y[mask]

print(f"Données nettoyées: {len(X)} échantillons")
print(f"Forme finale X: {X.shape}")
print(f"Forme finale y: {y.shape}")


Préparation des données d'entraînement...

Features utilisées: 9
Target: SiteEnergyUse(kWh)

Nettoyage des données...

Valeurs manquantes X: 0
Valeurs manquantes y: 0
Données nettoyées: 1277 échantillons
Forme finale X: (1277, 9)
Forme finale y: (1277,)


## 3️. Entraînement du Modèle Linear Regression

In [4]:
# Division des données
print("\n" + "=" * 100)
print("Division des données...")
print("=" * 100 + "\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train set: {X_train.shape[0]} échantillons")
print(f"Test set: {X_test.shape[0]} échantillons")

# Entraînement du modèle Linear Regression
print("\n" + "=" * 100)
print("Entraînement du modèle Linear Regression...")
print("=" * 100 + "\n")

model = LinearRegression()
model.fit(X_train, y_train)
print("Modèle entraîné avec succès!")

# Évaluation du modèle
print("\n" + "=" * 100)
print("Évaluation du modèle...")
print("=" * 100 + "\n")

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Calcul des métriques
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

# Affichage des résultats
print(f"Performance sur Train:")
print(f"   • R² Score: {r2_train:.4f} ({r2_train*100:.2f}%)")
print(f"   • MAE: {mae_train:,.0f}")
print(f"   • RMSE: {rmse_test:,.0f}")

print(f"\nPerformance sur Test:")
print(f"   • R² Score: {r2_test:.4f} ({r2_test*100:.2f}%)")
print(f"   • MAE: {mae_test:,.0f}")
print(f"   • RMSE: {rmse_test:,.0f}")

# Stockage des métriques pour BentoML
model_metrics = {
    "algorithm": "Linear Regression",
    "r2_train": float(r2_train),
    "r2_test": float(r2_test),
    "mae_train": float(mae_train),
    "mae_test": float(mae_test),
    "rmse_train": float(rmse_train),
    "rmse_test": float(rmse_test),
    "n_features": len(available_features),
    "n_train_samples": len(X_train),
    "n_test_samples": len(X_test)
}


Division des données...

Train set: 1021 échantillons
Test set: 256 échantillons

Entraînement du modèle Linear Regression...

Modèle entraîné avec succès!

Évaluation du modèle...

Performance sur Train:
   • R² Score: 0.9758 (97.58%)
   • MAE: 53,796
   • RMSE: 298,203

Performance sur Test:
   • R² Score: 0.9145 (91.45%)
   • MAE: 78,008
   • RMSE: 298,203


## 4️. Sauvegarde du Modèle avec BentoML

In [5]:
# 🗑️ Nettoyage des anciens modèles
print("\n" + "=" * 100)
print("Nettoyage des anciens modèles...")
print("=" * 100 + "\n")

try:
    # Lister tous les modèles existants avec le même nom
    existing_models = [m for m in bentoml.models.list() if m.tag.name == "seattle_energy_predictor"]
    
    if existing_models:
        print(f"Trouvé {len(existing_models)} ancien(s) modèle(s) à supprimer:")
        for old_model in existing_models:
            try:
                print(f"  Suppression: {old_model.tag}")
                bentoml.models.delete(old_model.tag)
                print(f"  Supprimé: {old_model.tag}")
            except Exception as e:
                print(f"  Échec suppression {old_model.tag}: {str(e)}")
    else:
        print("Aucun ancien modèle à nettoyer")
        
except Exception as e:
    print(f"Erreur lors du nettoyage: {str(e)}")
    print("Continuons avec la sauvegarde...")

# Préparation des métadonnées
metadata = {
    "project": "P6_Seattle_Energy_Prediction",
    "algorithm": "Linear Regression",
    "performance": {
        "r2_score": f"{r2_test:.4f} ({r2_test*100:.2f}%)",
        "mae_test": f"{mae_test:,.0f} kWh",
        "rmse_test": f"{rmse_test:,.0f} kWh"
    },
    "features": available_features,
    "target": target,
    "training_date": datetime.now().isoformat(),
    "model_version": "v1.0.0",
    "author": "ABAI_P6"
}

# Labels pour identification
labels = {
    "project": "seattle_energy_prediction",
    "algorithm": "linear_regression",
    "version": "1.0.0",
    "stage": "production"
}

print("\n" + "=" * 100)
print("Sauvegarde du nouveau modèle...")
print("=" * 100 + "\n")

# Sauvegarde avec BentoML
bentoml_model = bentoml.sklearn.save_model(
    name="seattle_energy_predictor",
    model=model,
    labels=labels,
    metadata=metadata,
    custom_objects={
        "feature_names": available_features,
        "model_metrics": model_metrics,
        "sample_input": X_test.iloc[0].to_dict()
    }
)

print(f"Modèle sauvegardé avec succès!")
print(f"Tag BentoML: {bentoml_model.tag}")
print(f"Path: {bentoml_model.path}")
print(f"Labels: {bentoml_model.info.labels}")


Nettoyage des anciens modèles...

Trouvé 1 ancien(s) modèle(s) à supprimer:
  Suppression: seattle_energy_predictor:d4bauf5wgk6kopak
  Supprimé: seattle_energy_predictor:d4bauf5wgk6kopak

Sauvegarde du nouveau modèle...

Modèle sauvegardé avec succès!
Tag BentoML: seattle_energy_predictor:62cwidvwgkllqpak
Path: C:\Users\aymer\AppData\Local\Temp\bentoml-model-seattle_energy_predictor-skuer3uj
Labels: {'project': 'seattle_energy_prediction', 'algorithm': 'linear_regression', 'version': '1.0.0', 'stage': 'production'}


## 5️. Vérification du Modèle Sauvegardé

In [6]:
# Vérification du modèle dans le store BentoML
print("\n" + "=" * 100)
print("Vérification du modèle sauvegardé...")
print("=" * 100 + "\n")

# Vérification basique du modèle
print(f"Vérification du modèle: {bentoml_model.tag}")
loaded_model = bentoml.models.get(bentoml_model.tag)
print("Modèle accessible dans le store BentoML!")

print(f"Labels: {loaded_model.info.labels}")
print(f"Créé le: {loaded_model.info.creation_time}")

print("\nLe modèle est correctement sauvegardé et accessible!")


Vérification du modèle sauvegardé...

Vérification du modèle: seattle_energy_predictor:62cwidvwgkllqpak
Modèle accessible dans le store BentoML!
Labels: {'project': 'seattle_energy_prediction', 'algorithm': 'linear_regression', 'version': '1.0.0', 'stage': 'production'}
Créé le: 2025-10-31 08:24:04.905020+00:00

Le modèle est correctement sauvegardé et accessible!
